# train1: temporary training notebook

Runs the two experiments in `RUNS` one after another, then releases the runtime. Re-running the cells after a disconnect resumes the unfinished run and skips finished ones. Delete this notebook when the results are in.

In [ ]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

REPO_URL = 'https://github.com/mericberktas/bitirme.git'
BRANCH = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/bitirme'
REPO_DIR = '/content/repo'

os.makedirs(DRIVE_ROOT, exist_ok=True)
# Models are downloaded to the LOCAL disk each session. Do not put the HF cache on Drive:
# symlinks and large files are not stored reliably there (it produced corrupted weights).
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch --quiet && git -C {REPO_DIR} checkout --quiet {BRANCH} && git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install --quiet -e .
!git log -1 --oneline

In [ ]:
# --- Make sure the environment is usable (safe to re-run; covers a restarted runtime) ---
import importlib.util, os, subprocess, sys

DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive/bitirme')
REPO_DIR = globals().get('REPO_DIR', '/content/repo')

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
if not os.path.isdir(f'{REPO_DIR}/src'):
    raise RuntimeError('Repo not found: run the Setup cell (section 1) first.')
os.chdir(REPO_DIR)  # config paths below are relative to the repo root

if importlib.util.find_spec('turkish_distill') is None:
    print('turkish_distill is not installed in this runtime: installing it now...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', REPO_DIR], check=True)
    importlib.invalidate_caches()
    if importlib.util.find_spec('turkish_distill') is None:
        sys.path.insert(0, f'{REPO_DIR}/src')
        os.environ['PYTHONPATH'] = f'{REPO_DIR}/src' + os.pathsep + os.environ.get('PYTHONPATH', '')

os.environ.setdefault('HF_HOME', '/content/hf_cache')  # local disk, never Drive
if 'HF_TOKEN' not in os.environ:
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception as exc:
        print('HF_TOKEN not set (only needed for gated models):', exc)

# Runs of this notebook, one after another (all: 1k student, 200K mix, 5400 steps, seed 42).
# (name, config, extra overrides). A later override wins, so train.lr below replaces the one in BASE.
BASE = ["data.corpus_path=/content/drive/MyDrive/bitirme/data/mix/mix_v2_200k.jsonl", "train.max_steps=5400", "train.lr=0.0001"]
RUNS = [
    ('scale_bge_mix200k_s5400_lr1e4', 'configs/stage1_single_bge_m3.yaml', []),
    ('scale_8k_mix200k_s5400_lr1e4', 'configs/stage1_single_modernbert_tr_embed.yaml', []),
]

import glob, os

for name, config, extra in RUNS:
    run_dirs = sorted(glob.glob(f'{DRIVE_ROOT}/runs/{name}/*/'))
    if any(os.path.isdir(d + 'final') for d in run_dirs):
        print(f'\n########## {name}: already finished, skipped')
        continue
    run_id_arg = f'--run-id {os.path.basename(run_dirs[-1].rstrip("/"))}' if run_dirs else ''   # resume after a disconnect
    overrides = ' '.join(f'--override {o}' for o in [f'name={name}'] + BASE + extra)
    print(f'\n########## {name} {run_id_arg or "(new run)"}')
    get_ipython().system(f'python -m turkish_distill.train --config {config} {run_id_arg} {overrides}')


In [ ]:
from google.colab import runtime
runtime.unassign()